In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from ipywidgets import interact, FloatSlider

# 解决中文显示与负号问题
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def simulate_membrane_instability(perturbation_field=0.0):
    # 1. 空间网格设置 (Monge 规范 xy 平面)
    L = 10.0  # 空间区域大小 (微米)
    N = 80    # 网格分辨率
    x = np.linspace(-L/2, L/2, N)
    y = np.linspace(-L/2, L/2, N)
    X, Y = np.meshgrid(x, y)
    
    # 2. 物理参数设置
    kappa = 1.0        # 归一化弯曲模量 (Bending Rigidity)
    gamma_0 = 0.1      # 基础张力
    gamma_lysis = 1.2  # 临界破裂张力阈值 (Lysis threshold)
    
    # 3. 构造高度场 h(x,y)
    p = perturbation_field
    
    w1, w2 = 1.2, 2.5  # 褶皱特征波数
    h_smooth = p * 0.15 * (np.sin(w1 * X) * np.cos(w1 * Y))
    h_wrinkle = (np.maximum(0, p - 0.4)**1.5) * 0.35 * (np.sin(w2 * X) * np.sin(w2 * Y) + np.cos(1.8 * w1 * X))
    
    Z = h_smooth + h_wrinkle
    
    # 4. 计算局部梯度 |grad h|^2 与诱导张力 gamma_induced
    dZx, dZy = np.gradient(Z, L/N, L/N)
    grad_sq = dZx**2 + dZy**2
    
    gamma_ind = gamma_0 + 0.8 * grad_sq
    
    # 5. 判别膜破裂 (Poration/Lysis)
    rupture_mask = gamma_ind > gamma_lysis
    Z_ruptured = Z.copy()
    Z_ruptured[rupture_mask] = np.nan

    # 6. 画布绘制
    fig = plt.figure(figsize=(14, 6), facecolor='#f8f9fa')
    
    # --- 左图：3D 膜形态物理演化 ---
    ax1 = fig.add_subplot(121, projection='3d', facecolor='#f8f9fa')
    ls = LightSource(azdeg=135, altdeg=45)
    
    rgb = ls.shade(grad_sq, cmap=plt.cm.plasma, vert_exag=0.2, blend_mode='soft')
    
    surf = ax1.plot_surface(X, Y, Z_ruptured, facecolors=rgb, rstride=1, cstride=1, 
                            linewidth=0, antialiased=True)
    ax1.set_xlim([-L/2, L/2])
    ax1.set_ylim([-L/2, L/2])
    ax1.set_zlim([-1.5, 1.5])
    ax1.view_init(elev=30, azim=45)
    ax1.axis('off')
    
    if p < 0.4:
        stage_title = "阶段 1：平整/微涨落状态 (Taut Phase)\n[系统能量稳定，抗弯曲项主导]"
    elif np.max(gamma_ind) <= gamma_lysis:
        stage_title = "阶段 2：屈曲褶皱状态 (Wrinkled Phase)\n[压应力突破屈曲阈值，出现周期性褶皱]"
    else:
        stage_title = "阶段 3：过度拉伸与破裂状态 (Lysis/Poration Phase)\n[局部张力突破阈值，膜发生穿孔解体!]"

    ax1.set_title(f"3D 膜构型 (微扰场强度 P = {p:.2f})\n{stage_title}", fontsize=11, fontweight='bold')

    # --- 右图：2D 局部张力分布图 & 破裂区域 ---
    ax2 = fig.add_subplot(122)
    im = ax2.imshow(gamma_ind, extent=[-L/2, L/2, -L/2, L/2], origin='lower', cmap='YlOrRd', vmin=0.1, vmax=1.5)
    
    if np.any(rupture_mask):
        ax2.contour(X, Y, rupture_mask, levels=[0.5], colors='black', linewidths=2)
        ax2.text(0, 0, "膜发生穿孔/破裂!", color='darkred', fontsize=12, fontweight='bold', 
                 ha='center', va='center', bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="red", lw=2))
        
    cbar = fig.colorbar(im, ax=ax2, shrink=0.7)
    # ⚡ 加上 r 引号修复警告
    cbar.set_label(r'局部诱导张力 $\gamma_{ind}$ (mN/m)', fontsize=10)
    
    ax2.set_title("2D 面张力分布图 (黑色圈定破裂区域)", fontsize=11, fontweight='bold')
    # ⚡ 加上 r 引号修复警告
    ax2.set_xlabel(r"x ($\mu m$)")
    ax2.set_ylabel(r"y ($\mu m$)")
    
    plt.tight_layout()
    plt.show()

interact(
    simulate_membrane_instability,
    perturbation_field=FloatSlider(
        min=0.0, max=1.5, step=0.05, value=0.1, 
        continuous_update=False, description='微扰场强度 P'
    )
);

interactive(children=(FloatSlider(value=0.1, continuous_update=False, description='微扰场强度 P', max=1.5, step=0.0…